# 31 cellrank exploratory

**Purpose:** Run the exploratory, memory-safe CellRank fate-mapping analysis.

**Original analysis notebook:** `GBM_CellRank_malignant_fate_mapping_v2_MEMORY_SAFE.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# Exploratory CellRank fate mapping of malignant glioma states

This notebook reproduces the exploratory CellRank analysis reported in
Supplementary Fig. S9.

The analysis:

- restricts the integrated object to malignant-state cells;
- draws a stratified 20,000-cell subset for memory-safe GPCCA fitting;
- constructs a 30-nearest-neighbor graph from `X_scVI`;
- calculates diffusion pseudotime;
- defines the transition matrix as
  `0.9 × PseudotimeKernel + 0.1 × ConnectivityKernel`;
- fits GPCCA with 12 Schur components and 6 macrostates;
- exports pseudotime, macrostate, terminal-state, and fate-probability outputs;
- generates the panels needed for Supplementary Fig. S9.

This is an exploratory phenotypic-organization analysis. It is not RNA
velocity, lineage tracing, or evidence of irreversible differentiation.

## Input configuration

By default, the notebook expects the annotated integrated object at:

`data/processed/integrated/GBM_core_4datasets_final_annotation_malignant_states.h5ad`

The path can be overridden without editing the notebook by setting:

- `GLIOMA_PROJECT_ROOT`
- `GLIOMA_INPUT_H5AD`
- `GLIOMA_OUTPUT_DIR`

In [ ]:
import gc
import json
import os
import re
import sys
import traceback
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")

import anndata as ad
import cellrank as cr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
from IPython.display import display

print("Python:", sys.version)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("cellrank:", cr.__version__)
print("scvelo:", scv.__version__)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white")

In [ ]:
# ===== GLOBAL CONFIGURATION: MEMORY-SAFE VERSION =====

def find_repository_root(start: Path) -> Path:
    # Return the nearest parent containing both README.md and code/.
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GLIOMA_INPUT_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "integrated"
        / "GBM_core_4datasets_final_annotation_malignant_states.h5ad",
    )
).expanduser().resolve()

ANALYSIS_NAME = "cellrank_malignant_fate_mapping_v2_memory_safe"

OUTDIR = Path(
    os.environ.get(
        "GLIOMA_OUTPUT_DIR",
        PROJECT_ROOT / "results" / ANALYSIS_NAME,
    )
).expanduser().resolve()

FIGDIR = OUTDIR / "figures"
TABLEDIR = OUTDIR / "tables"
LOGDIR = OUTDIR / "logs"

for directory in (OUTDIR, FIGDIR, TABLEDIR, LOGDIR):
    directory.mkdir(parents=True, exist_ok=True)

FINAL_H5AD_NAME = (
    "GBM_malignant_CellRank_PseudotimeKernel_"
    "v2_memory_safe_result.h5ad"
)
FINAL_H5AD = OUTDIR / FINAL_H5AD_NAME

RANDOM_SEED = 13

# GPCCA can require a dense matrix when PETSc/SLEPc is unavailable.
# A stratified 20,000-cell subset keeps the dense Brandts calculation
# within a practical memory footprint.
MAX_CELLS_FOR_CR = 20_000

USE_REP = "X_scVI"
N_NEIGHBORS = 30
RECOMPUTE_MALIGNANT_UMAP = True

ROOT_SCORE_KEY = "cr_root_score"
PSEUDOTIME_KEY = "cr_dpt_pseudotime"
MAX_MACROSTATE_MEMBERSHIP_KEY = "cr_max_macrostate_membership"
MAX_FATE_PROBABILITY_KEY = "cr_max_fate_probability"

N_SCHUR_COMPONENTS = 12
N_MACROSTATES = 6

USE_COMBINED_KERNEL = True
PSEUDOTIME_KERNEL_WEIGHT = 0.9
CONNECTIVITY_KERNEL_WEIGHT = 0.1

RUN_LINEAGE_DRIVERS = False
SAVE_FINAL_H5AD = True

run_info = {
    "analysis": ANALYSIS_NAME,
    "started_at": datetime.now().isoformat(timespec="seconds"),
    "input_h5ad": str(INPUT_H5AD),
    "random_seed": RANDOM_SEED,
    "max_cells_for_cr": MAX_CELLS_FOR_CR,
    "use_rep": USE_REP,
    "n_neighbors": N_NEIGHBORS,
    "n_schur_components": N_SCHUR_COMPONENTS,
    "n_macrostates": N_MACROSTATES,
    "pseudotime_kernel_weight": PSEUDOTIME_KERNEL_WEIGHT,
    "connectivity_kernel_weight": CONNECTIVITY_KERNEL_WEIGHT,
}


def check_config_is_defined():
    required = [
        "PROJECT_ROOT",
        "INPUT_H5AD",
        "OUTDIR",
        "FIGDIR",
        "TABLEDIR",
        "LOGDIR",
        "FINAL_H5AD",
        "RANDOM_SEED",
        "MAX_CELLS_FOR_CR",
        "USE_REP",
        "N_NEIGHBORS",
        "RECOMPUTE_MALIGNANT_UMAP",
        "ROOT_SCORE_KEY",
        "PSEUDOTIME_KEY",
        "MAX_MACROSTATE_MEMBERSHIP_KEY",
        "MAX_FATE_PROBABILITY_KEY",
        "N_SCHUR_COMPONENTS",
        "N_MACROSTATES",
        "USE_COMBINED_KERNEL",
        "PSEUDOTIME_KERNEL_WEIGHT",
        "CONNECTIVITY_KERNEL_WEIGHT",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise NameError(
            "Missing configuration variables. Re-run this cell. Missing: "
            + ", ".join(missing)
        )


def dense_memory_gib(n_cells, dtype_bytes=8):
    return (n_cells**2 * dtype_bytes) / (1024**3)


check_config_is_defined()

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_H5AD)
print("Input exists:", INPUT_H5AD.exists())
print("Output:", OUTDIR)
print("Final h5ad:", FINAL_H5AD)
print("MAX_CELLS_FOR_CR:", MAX_CELLS_FOR_CR)
print("USE_REP:", USE_REP)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Input AnnData object was not found: {INPUT_H5AD}\n"
        "Set GLIOMA_INPUT_H5AD to the correct file path."
    )

In [ ]:
def now():
    return datetime.now().isoformat(timespec="seconds")

def savefig(path, dpi=300):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()

def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def truthy_mask(s):
    x = s.astype(str).str.strip().str.lower()
    return x.isin(["true", "1", "yes", "y", "t", "malignant", "tumor", "tumour"])

def choose_malignant_mask(obs):
    if "is_malignant_state_analysis_cell" in obs.columns:
        return truthy_mask(obs["is_malignant_state_analysis_cell"])

    for c in ["final_compartment_major", "final_compartment_simple"]:
        if c in obs.columns:
            x = obs[c].astype(str).str.lower()
            if x.str.contains("malig|tumor|tumour", regex=True).any():
                return x.str.contains("malig|tumor|tumour", regex=True)

    if "malignant_state" in obs.columns:
        x = obs["malignant_state"].astype(str).str.lower()
        bad = ["nan", "na", "none", "unknown", "unassigned", "", "non-malignant", "non_malignant"]
        return ~x.isin(bad)

    raise ValueError("Could not infer malignant-cell mask.")

def zscore_numeric(s):
    x = pd.to_numeric(s, errors="coerce")
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.zeros(len(x)), index=s.index)
    return pd.Series((x - mu) / sd, index=s.index)

def make_categorical(adata, keys):
    for k in keys:
        if k in adata.obs.columns:
            adata.obs[k] = adata.obs[k].astype(str).fillna("NA").astype("category")

def plot_umap(adata, key, fname, title=None, size=4):
    if key not in adata.obs.columns and key not in adata.var_names:
        print("Skip missing key:", key)
        return
    sc.pl.umap(
        adata,
        color=key,
        frameon=False,
        size=size,
        title=title if title else key,
        show=False,
    )
    savefig(FIGDIR / fname)

def export_value_counts(adata, keys, prefix):
    rows = []
    for key in keys:
        if key in adata.obs.columns:
            vc = adata.obs[key].astype(str).value_counts(dropna=False)
            tmp = vc.rename("n_cells").reset_index()
            tmp.columns = [key, "n_cells"]
            tmp.insert(0, "field", key)
            rows.append(tmp)
    if rows:
        out = pd.concat(rows, ignore_index=True)
        out.to_csv(TABLEDIR / f"{prefix}_value_counts.tsv", sep="\t", index=False)
        return out
    return pd.DataFrame()

def stratified_subsample_adata(adata, max_cells, strata_cols, seed=13):
    if max_cells is None or adata.n_obs <= max_cells:
        return adata, False

    rng = np.random.default_rng(seed)
    available = [c for c in strata_cols if c in adata.obs.columns]

    if not available:
        selected = rng.choice(adata.obs_names.to_numpy(), size=max_cells, replace=False)
        return adata[selected].copy(), True

    strata = adata.obs[available].astype(str).agg("|".join, axis=1)
    frac = max_cells / adata.n_obs
    selected = []

    for _, idx in strata.groupby(strata).groups.items():
        idx = np.array(list(idx))
        n = max(1, int(round(len(idx) * frac)))
        n = min(n, len(idx))
        selected.extend(rng.choice(idx, size=n, replace=False).tolist())

    if len(selected) > max_cells:
        selected = rng.choice(np.array(selected), size=max_cells, replace=False).tolist()

    return adata[selected].copy(), True

run_info = {
    "analysis": ANALYSIS_NAME,
    "started_at": now(),
    "input_h5ad": str(INPUT_H5AD),
    "random_seed": RANDOM_SEED,
}

def lineage_object_to_dataframe(obj, index):
    # Convert a CellRank Lineage-like object to a numeric DataFrame.
    if obj is None:
        return None

    if hasattr(obj, "to_df"):
        frame = obj.to_df()
    else:
        array = obj.toarray() if hasattr(obj, "toarray") else np.asarray(obj)
        if array.ndim != 2:
            raise ValueError(f"Expected a 2-D lineage matrix, got shape {array.shape}")
        names = getattr(obj, "names", None)
        columns = list(names) if names is not None else [
            f"state_{i + 1}" for i in range(array.shape[1])
        ]
        frame = pd.DataFrame(array, index=index, columns=columns)

    frame = frame.reindex(index)
    return frame.apply(pd.to_numeric, errors="coerce")


def export_umap_value(adata, key, filename, title):
    # Save a UMAP panel for a categorical or continuous obs variable.
    if key not in adata.obs:
        raise KeyError(f"{key!r} is not present in adata.obs")
    sc.pl.umap(
        adata,
        color=key,
        frameon=False,
        title=title,
        size=5,
        show=False,
    )
    savefig(FIGDIR / filename)

In [ ]:
# Load in backed mode and subset malignant cells
adata_b = sc.read_h5ad(INPUT_H5AD, backed="r")
print("Full object:", adata_b.shape)

obs = adata_b.obs.copy()
mal_mask = choose_malignant_mask(obs)

print("Selected malignant cells:", int(mal_mask.sum()))
print("Other cells:", int((~mal_mask).sum()))

adata = adata_b[mal_mask.values].to_memory()

try:
    adata_b.file.close()
except Exception:
    pass

adata.var_names_make_unique()

print("Malignant object:", adata.shape)
print("obsm:", list(adata.obsm.keys()))
print("layers:", list(adata.layers.keys()))
print("obsp:", list(adata.obsp.keys()))

run_info["n_cells_malignant_initial"] = int(adata.n_obs)
run_info["n_genes_initial"] = int(adata.n_vars)

In [ ]:
# ===== Memory-safe stratified downsampling BEFORE graph/CellRank =====

check_config_is_defined()

print("Before downsampling:", adata.shape)

adata, was_downsampled = stratified_subsample_adata(
    adata,
    max_cells=MAX_CELLS_FOR_CR,
    strata_cols=["malignant_state", "source_dataset", "core_dataset", "patient_id", "sample_id"],
    seed=RANDOM_SEED,
)

gc.collect()

print("Downsampled:", was_downsampled)
print("After downsampling:", adata.shape)
print(f"Estimated dense matrix memory for GPCCA Brandts: {dense_memory_gib(adata.n_obs):.2f} GiB")

if adata.n_obs > 30000:
    raise MemoryError(
        f"adata has {adata.n_obs} cells. Too large for dense Brandts method. "
        "Set MAX_CELLS_FOR_CR to 20000 or install working PETSc/SLEPc."
    )

make_categorical(
    adata,
    [
        "malignant_state",
        "source_dataset",
        "core_dataset",
        "dataset_id",
        "patient_id",
        "sample_id",
        "condition",
        "final_compartment_major",
        "final_compartment_simple",
        "final_annotation_refined",
        "final_cnv_consensus_by_cluster",
        "copykat_pred_clean",
    ],
)

vc = export_value_counts(
    adata,
    [
        "malignant_state",
        "source_dataset",
        "core_dataset",
        "dataset_id",
        "patient_id",
        "sample_id",
        "condition",
        "final_cnv_consensus_by_cluster",
        "copykat_pred_clean",
    ],
    "cellrank_input",
)

run_info["was_downsampled"] = bool(was_downsampled)
run_info["n_cells_for_cellrank"] = int(adata.n_obs)
run_info["n_genes_for_cellrank"] = int(adata.n_vars)
run_info["estimated_dense_memory_gib"] = float(dense_memory_gib(adata.n_obs))

display(vc.head(50))

In [ ]:
# Prepare X for driver analysis
# CellRank PseudotimeKernel itself uses graph + pseudotime.
# For downstream driver genes, make X = log1p-normalized counts if counts layer exists.

if "counts" in adata.layers:
    print("Using adata.layers['counts'] -> normalize_total + log1p for adata.X")
    adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
else:
    print("No counts layer found. Using existing adata.X.")

adata.uns["cellrank_expression_preprocessing"] = {
    "note": "X used for downstream lineage-driver expression. PseudotimeKernel uses graph+pseudotime.",
    "created_at": now(),
}

In [ ]:
# ===== Build malignant-only graph on X_scVI =====

check_config_is_defined()

if USE_REP not in adata.obsm:
    raise KeyError(f"{USE_REP} not found. Available obsm keys: {list(adata.obsm.keys())}")

if "X_umap" in adata.obsm:
    adata.obsm["X_umap_fullTME"] = adata.obsm["X_umap"].copy()

print("Computing neighbors on malignant-only subset...")
sc.pp.neighbors(
    adata,
    n_neighbors=N_NEIGHBORS,
    use_rep=USE_REP,
    random_state=RANDOM_SEED,
)

if RECOMPUTE_MALIGNANT_UMAP:
    print("Computing malignant-only UMAP...")
    sc.tl.umap(adata, random_state=RANDOM_SEED, min_dist=0.3)

print("obsp after neighbors:", list(adata.obsp.keys()))
print("uns['neighbors']:", adata.uns.get("neighbors", {}).keys())

run_info["use_rep"] = USE_REP
run_info["n_neighbors"] = N_NEIGHBORS
run_info["recompute_malignant_umap"] = RECOMPUTE_MALIGNANT_UMAP

In [ ]:
# Basic malignant landscape plots

basic_keys = [
    "malignant_state",
    "source_dataset",
    "core_dataset",
    "condition",
    "patient_id",
    "final_cnv_consensus_by_cluster",
    "copykat_pred_clean",
    "score_AC_like_reactive_program",
    "score_MES_like_program",
    "score_NPC_neural_program",
    "score_OPC_proneural_program",
    "score_cycling_program",
    "score_hypoxia_stress_program",
]

for key in basic_keys:
    if key in adata.obs.columns:
        plot_umap(adata, key, f"UMAP_{key}.png")

print("Basic figures saved:", FIGDIR)

In [ ]:
# Root score for DPT pseudotime

ROOT_POSITIVE_SCORE_COLS = [
    "score_OPC_proneural_program",
    "score_NPC_neural_program",
    "score_cycling_program",
]

ROOT_NEGATIVE_SCORE_COLS = [
    "score_MES_like_program",
    "score_AC_like_reactive_program",
    "score_hypoxia_stress_program",
]

pos_cols = [c for c in ROOT_POSITIVE_SCORE_COLS if c in adata.obs.columns]
neg_cols = [c for c in ROOT_NEGATIVE_SCORE_COLS if c in adata.obs.columns]

print("Root positive score columns:", pos_cols)
print("Root negative score columns:", neg_cols)

if pos_cols:
    pos = pd.concat([zscore_numeric(adata.obs[c]) for c in pos_cols], axis=1).mean(axis=1)
else:
    marker_genes = [g for g in ["SOX2", "OLIG2", "PDGFRA", "DLL3", "MKI67", "TOP2A"] if g in adata.var_names]
    if not marker_genes:
        raise ValueError("No root-score columns or fallback marker genes found.")
    sc.tl.score_genes(adata, marker_genes, score_name="fallback_root_marker_score")
    pos = zscore_numeric(adata.obs["fallback_root_marker_score"])

if neg_cols:
    neg = pd.concat([zscore_numeric(adata.obs[c]) for c in neg_cols], axis=1).mean(axis=1)
else:
    neg = pd.Series(np.zeros(adata.n_obs), index=adata.obs_names)

adata.obs[ROOT_SCORE_KEY] = pos - 0.25 * neg

root_ix = int(np.nanargmax(pd.to_numeric(adata.obs[ROOT_SCORE_KEY], errors="coerce").values))
adata.uns["iroot"] = root_ix

print("DPT root index:", root_ix)
print("DPT root cell:", adata.obs_names[root_ix])

if "malignant_state" in adata.obs.columns:
    print("DPT root malignant_state:", adata.obs.iloc[root_ix]["malignant_state"])

plot_umap(adata, ROOT_SCORE_KEY, "UMAP_cr_root_score.png", "DPT root score")

write_json(
    {
        "root_ix": root_ix,
        "root_cell": str(adata.obs_names[root_ix]),
        "positive_columns": pos_cols,
        "negative_columns": neg_cols,
    },
    TABLEDIR / "root_score_summary.json",
)

In [ ]:
# DPT pseudotime

sc.tl.diffmap(adata, n_comps=15)
sc.tl.dpt(adata, n_dcs=10)

adata.obs[PSEUDOTIME_KEY] = pd.to_numeric(adata.obs["dpt_pseudotime"], errors="coerce")

bad = ~np.isfinite(adata.obs[PSEUDOTIME_KEY].values)
print("Non-finite pseudotime cells:", int(bad.sum()))

if bad.sum() > 0:
    adata.obs.loc[bad, PSEUDOTIME_KEY] = np.nanmedian(adata.obs[PSEUDOTIME_KEY].values)

plot_umap(adata, PSEUDOTIME_KEY, "UMAP_cr_dpt_pseudotime.png", "DPT pseudotime")

pt_df = pd.DataFrame({
    "pseudotime": adata.obs[PSEUDOTIME_KEY],
    "root_score": adata.obs[ROOT_SCORE_KEY],
}, index=adata.obs_names)

for k in ["malignant_state", "source_dataset", "patient_id", "sample_id", "condition"]:
    if k in adata.obs.columns:
        pt_df[k] = adata.obs[k].astype(str).values

pt_df.to_csv(TABLEDIR / "cellrank_input_pseudotime_per_cell.tsv", sep="\t")

if "malignant_state" in pt_df.columns:
    state_pt = (
        pt_df.groupby("malignant_state")
        .agg(
            n_cells=("pseudotime", "size"),
            pseudotime_median=("pseudotime", "median"),
            pseudotime_mean=("pseudotime", "mean"),
            root_score_median=("root_score", "median"),
        )
        .sort_values("pseudotime_median")
    )
    state_pt.to_csv(TABLEDIR / "pseudotime_by_malignant_state.tsv", sep="\t")
    display(state_pt)

run_info["pseudotime_key"] = PSEUDOTIME_KEY
run_info["root_score_key"] = ROOT_SCORE_KEY

In [ ]:
# CellRank PseudotimeKernel

pk = cr.kernels.PseudotimeKernel(adata, time_key=PSEUDOTIME_KEY)
pk.compute_transition_matrix(threshold_scheme="soft")

kernel = pk
kernel_description = "PseudotimeKernel"

if USE_COMBINED_KERNEL:
    ck = cr.kernels.ConnectivityKernel(adata)
    ck.compute_transition_matrix()
    kernel = PSEUDOTIME_KERNEL_WEIGHT * pk + CONNECTIVITY_KERNEL_WEIGHT * ck
    kernel_description = f"{PSEUDOTIME_KERNEL_WEIGHT}*PseudotimeKernel + {CONNECTIVITY_KERNEL_WEIGHT}*ConnectivityKernel"

print("Kernel:", kernel_description)

run_info["kernel_description"] = kernel_description

try:
    pk.plot_random_walks(
        start_ixs={PSEUDOTIME_KEY: [0, 0.05]},
        basis="umap",
        seed=RANDOM_SEED,
        size=4,
    )
    savefig(FIGDIR / "CellRank_random_walks_from_low_pseudotime.png")
except Exception as e:
    print("Random-walk plot failed; continuing.")
    print(repr(e))

In [ ]:
# ===== GPCCA macrostates and terminal states - MEMORY SAFE =====

print("Starting GPCCA...")
print("Current adata shape:", adata.shape)
print(f"Estimated dense matrix memory if Brandts is used: {dense_memory_gib(adata.n_obs):.2f} GiB")

estimator = cr.estimators.GPCCA(kernel)

try:
    import petsc4py
    import slepc4py
    SCHUR_METHOD = "krylov"
    print("PETSc/SLEPc available. Using sparse Krylov method.")
except Exception:
    SCHUR_METHOD = "brandts"
    print("PETSc/SLEPc not available. Using dense Brandts method.")
    if adata.n_obs > 30000:
        raise MemoryError(
            f"Too many cells for dense Brandts method: {adata.n_obs} cells. "
            "Set MAX_CELLS_FOR_CR to 20000 or install working petsc4py/slepc4py."
        )

try:
    estimator.compute_schur(
        n_components=N_SCHUR_COMPONENTS,
        method=SCHUR_METHOD,
    )
except TypeError:
    print("This CellRank version did not accept method argument. Retrying compute_schur without method.")
    print("WARNING: This may use the default method and may need more memory.")
    estimator.compute_schur(n_components=N_SCHUR_COMPONENTS)

try:
    estimator.plot_spectrum(real_only=True)
    savefig(FIGDIR / "CellRank_GPCCA_spectrum.png")
except Exception as e:
    print("Spectrum plot failed; continuing.")
    print(repr(e))

cluster_key = "malignant_state" if "malignant_state" in adata.obs.columns else None

print("Computing macrostates...")
if cluster_key:
    estimator.compute_macrostates(N_MACROSTATES, cluster_key=cluster_key)
else:
    estimator.compute_macrostates(N_MACROSTATES)

try:
    estimator.plot_macrostates(
        which="all",
        basis="umap",
        legend_loc="right",
        title="CellRank macrostates",
        size=4,
    )
    savefig(FIGDIR / "CellRank_macrostates_all.png")
except Exception as e:
    print("Macrostate plot failed; continuing.")
    print(repr(e))

print("Predicting terminal states...")
estimator.predict_terminal_states()

try:
    estimator.plot_macrostates(
        which="terminal",
        basis="umap",
        legend_loc="right",
        title="CellRank terminal states",
        size=4,
    )
    savefig(FIGDIR / "CellRank_terminal_states.png")
except Exception as e:
    print("Terminal-state plot failed; continuing.")
    print(repr(e))

state_cols = [c for c in adata.obs.columns if "term_states" in c or "macro" in c.lower()]
print("CellRank obs columns:", state_cols)

if state_cols:
    adata.obs[state_cols].to_csv(
        TABLEDIR / "cellrank_macrostates_terminal_states_obs.tsv",
        sep="\t",
    )

run_info["n_schur_components"] = N_SCHUR_COMPONENTS
run_info["n_macrostates"] = N_MACROSTATES
run_info["schur_method"] = SCHUR_METHOD

print("GPCCA done.")

In [ ]:
# ===== Fate probabilities and Supplementary Fig. S9 panels =====

estimator.compute_fate_probabilities()

fate_df = lineage_object_to_dataframe(
    estimator.fate_probabilities,
    adata.obs_names,
)

if fate_df is None or fate_df.empty:
    raise RuntimeError("CellRank fate probabilities could not be exported.")

fate_df.to_csv(
    TABLEDIR / "cellrank_fate_probabilities.tsv",
    sep="\t",
)
print("Fate probabilities:", fate_df.shape)
display(fate_df.head())

# Panel E: maximum GPCCA macrostate-membership probability per cell.
macrostate_memberships = lineage_object_to_dataframe(
    getattr(estimator, "macrostates_memberships", None),
    adata.obs_names,
)

if macrostate_memberships is None or macrostate_memberships.empty:
    candidate_keys = [
        key
        for key in adata.obsm
        if "macrostate" in key.lower() and "membership" in key.lower()
    ]
    if not candidate_keys:
        raise RuntimeError(
            "GPCCA macrostate-membership probabilities were not available."
        )
    macrostate_memberships = lineage_object_to_dataframe(
        adata.obsm[candidate_keys[0]],
        adata.obs_names,
    )

macrostate_memberships.to_csv(
    TABLEDIR / "cellrank_macrostate_memberships.tsv",
    sep="\t",
)

adata.obs[MAX_MACROSTATE_MEMBERSHIP_KEY] = (
    macrostate_memberships.max(axis=1).astype(float)
)

export_umap_value(
    adata,
    MAX_MACROSTATE_MEMBERSHIP_KEY,
    "S9E_UMAP_maximum_macrostate_membership_probability.png",
    "Maximum GPCCA macrostate-membership probability",
)

# Panel F: maximum fate probability grouped by malignant-state label.
adata.obs[MAX_FATE_PROBABILITY_KEY] = fate_df.max(axis=1).astype(float)

panel_f_columns = [MAX_FATE_PROBABILITY_KEY]
if "malignant_state" in adata.obs:
    panel_f_columns.insert(0, "malignant_state")

panel_f_data = adata.obs[panel_f_columns].copy()
panel_f_data.to_csv(
    TABLEDIR / "S9F_maximum_fate_probability_by_malignant_state.tsv",
    sep="\t",
)

if "malignant_state" not in panel_f_data:
    raise KeyError("malignant_state is required for Supplementary Fig. S9F.")

panel_f_data["malignant_state"] = panel_f_data["malignant_state"].astype(str)
state_order = (
    panel_f_data.groupby("malignant_state")[MAX_FATE_PROBABILITY_KEY]
    .median()
    .sort_values(ascending=False)
    .index
    .tolist()
)

values_by_state = [
    panel_f_data.loc[
        panel_f_data["malignant_state"] == state,
        MAX_FATE_PROBABILITY_KEY,
    ].dropna().to_numpy()
    for state in state_order
]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.boxplot(
    values_by_state,
    labels=state_order,
    showfliers=False,
    widths=0.55,
)

rng = np.random.default_rng(RANDOM_SEED)
for position, values in enumerate(values_by_state, start=1):
    jitter = rng.normal(loc=position, scale=0.045, size=len(values))
    ax.scatter(jitter, values, s=4, alpha=0.12, linewidths=0)

ax.set_ylabel("Maximum fate probability")
ax.set_xlabel("Malignant-state annotation")
ax.set_ylim(0, 1.02)
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(
    FIGDIR / "S9F_maximum_fate_probability_by_malignant_state.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

# Panel G: cell-wise fate probability toward each terminal macrostate.
for column in fate_df.columns:
    key = "fate_probability_" + re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(column),
    ).strip("_")
    adata.obs[key] = fate_df[column].astype(float)
    export_umap_value(
        adata,
        key,
        f"S9G_UMAP_{key}.png",
        str(column),
    )

try:
    estimator.plot_fate_probabilities(
        same_plot=False,
        basis="umap",
        ncols=3,
        size=4,
    )
    savefig(FIGDIR / "S9G_CellRank_fate_probabilities_all_terminal_states.png")
except Exception as exc:
    print(
        "CellRank's combined fate-probability plot failed; "
        "per-state UMAPs were saved."
    )
    print(repr(exc))

try:
    cr.pl.circular_projection(
        adata,
        keys="malignant_state" if "malignant_state" in adata.obs.columns else None,
        legend_loc="right",
    )
    savefig(FIGDIR / "CellRank_circular_projection_malignant_state.png")
except Exception as exc:
    print("Circular projection failed; continuing.")
    print(repr(exc))

In [ ]:
# Lineage drivers

if RUN_LINEAGE_DRIVERS:
    try:
        if "malignant_state" in adata.obs.columns:
            drivers = estimator.compute_lineage_drivers(
                return_drivers=True,
                cluster_key="malignant_state",
            )
        else:
            drivers = estimator.compute_lineage_drivers(return_drivers=True)

        drivers.to_csv(TABLEDIR / "cellrank_lineage_drivers_all.tsv", sep="\t")
        print("Drivers:", drivers.shape)
        display(drivers.head(30))

        genes = drivers.index.astype(str)
        exclude = (
            genes.str.startswith("MT-")
            | genes.str.startswith("MT.")
            | genes.str.startswith("RPL")
            | genes.str.startswith("RPS")
            | genes.str.match(r"^HB[ABDEGQMZ]")
        )

        filtered = drivers.loc[~exclude].copy()
        filtered.to_csv(TABLEDIR / "cellrank_lineage_drivers_filtered_no_mt_ribo_hb.tsv", sep="\t")

        numeric_cols = filtered.select_dtypes(include=[np.number]).columns.tolist()
        top_rows = []

        for col in numeric_cols:
            if "corr" in str(col).lower():
                tmp = filtered[[col]].dropna().sort_values(col, ascending=False).head(50).copy()
                tmp["gene"] = tmp.index
                tmp["driver_metric"] = col
                tmp["rank"] = np.arange(1, tmp.shape[0] + 1)
                top_rows.append(tmp.reset_index(drop=True))

        if top_rows:
            top_drivers = pd.concat(top_rows, ignore_index=True)
            top_drivers.to_csv(TABLEDIR / "cellrank_top50_drivers_per_lineage.tsv", sep="\t", index=False)
            display(top_drivers.head(30))

    except Exception:
        print("Lineage driver analysis failed.")
        print(traceback.format_exc())
else:
    print("RUN_LINEAGE_DRIVERS=False")

In [ ]:
# Save final outputs

export_cols = []

for c in [
    "malignant_state",
    "source_dataset",
    "core_dataset",
    "dataset_id",
    "patient_id",
    "sample_id",
    "condition",
    "final_compartment_major",
    "final_compartment_simple",
    "final_annotation_refined",
    "final_cnv_consensus_by_cluster",
    "copykat_pred_clean",
    ROOT_SCORE_KEY,
    PSEUDOTIME_KEY,
]:
    if c in adata.obs.columns:
        export_cols.append(c)

for c in adata.obs.columns:
    if "term_states" in c or "macro" in c.lower() or "lineage" in c.lower():
        if c not in export_cols:
            export_cols.append(c)

adata.obs[export_cols].to_csv(TABLEDIR / "cellrank_malignant_per_cell_metadata.tsv", sep="\t")

run_info["finished_at"] = now()
run_info["output_dir"] = str(OUTDIR)
run_info["figures_dir"] = str(FIGDIR)
run_info["tables_dir"] = str(TABLEDIR)
run_info["final_n_cells"] = int(adata.n_obs)
run_info["final_n_genes"] = int(adata.n_vars)

write_json(run_info, LOGDIR / "cellrank_run_info.json")

if SAVE_FINAL_H5AD:
    print("Writing:", FINAL_H5AD)
    adata.write_h5ad(FINAL_H5AD, compression="gzip")

print("DONE")
print("Figures:", FIGDIR)
print("Tables:", TABLEDIR)
print("Logs:", LOGDIR)

## Supplementary Fig. S9 output mapping

After successful execution, the `figures/` directory contains:

- **S9A** — `UMAP_malignant_state.png`
- **S9B** — `UMAP_cr_dpt_pseudotime.png`
- **S9C** — `UMAP_cr_root_score.png`
- **S9D** — `CellRank_terminal_states.png`
- **S9E** — `S9E_UMAP_maximum_macrostate_membership_probability.png`
- **S9F** — `S9F_maximum_fate_probability_by_malignant_state.png`
- **S9G** — per-terminal-state files beginning with `S9G_UMAP_fate_probability_`

The corresponding numeric tables are written to `tables/`.